# 啤酒数据：数据预处理

目标：为三个问题的输入进行数据预处理工作。
输出：
- chem_log10_matrix_v1.csv：250×231 主矩阵，half_min 填充，无 NA
- chem_log10_matrix_lod_sqrt2_proxy_v1.csv / chem_log10_matrix_qrilc_v1.csv：其他填充
- chem_log10_matrix_v1_manifest.json：填充规则记录

- chem_nondetect_sensitivity_v1.csv：左删失敏感性
- chem_outlier_unified_flags_v1.csv：异常值标记

In [2]:
from pathlib import Path
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams["font.sans-serif"] = ["Hiragino Sans GB", "Arial Unicode MS"]  
plt.rcParams["axes.unicode_minus"] = False 

from scipy import stats


from IPython.display import display, Markdown
from scipy import stats
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid")
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
assert RAW_DIR.exists(), f"未找到原始数据目录: {RAW_DIR}"
print(f"项目目录: {PROJECT_ROOT}")

项目目录: /Users/thomasbuch/Documents/work_for_me/public


## 1. 数据结构

表格 1 包含 250 种啤酒、231 个化学特征，另有 3 个标识/分类字段。
来自01文件的数据分类结果显示：

| 类别 | 数量 | 当前处理原则 |
| --- | ---: | --- |
| 标准理化指标 | 16 | 文件 1 已采用常用对数，保留其原单位下的 log10 值 |
| 有标定浓度的化合物 | 51 | 按文件 8 的实际变换关系还原浓度，统一为 mg/L 后再取 log10 |
| 文件 8 中出现但无校准标记 | 3 | 暂按半定量/A.U.变量保留，不解释为 mg/L |
| 无文件 8 标定浓度的 GC-MS 相对丰度 | 157 | 保留原有对数峰面积尺度；共同底限按左删失处理 |
| 派生变量 | 4 | 与原始化合物分开；`aroma_hops_sum` 单独验证和回填 |

因此化学特征总数为 16 + 51 + 3 + 157 + 4 = 231。

### 缺失与未检出

- 文件 1 的 60 个普通化合物 NA 在文件 8 对应位置全部为 0。这些值只能判为“未检出/左删失”，不能当作观测到的零浓度或真实浓度回填。
- 对这 60 个单元格建立未检出掩码，比较半最小正值、LOD/√2 代理和 QRILC 左删失填补。主分析采用与现有数据底限一致的半最小正值，另外两种用于敏感性分析。
- `aroma_hops_sum` 的 13 个 NA 属于派生变量缺失。用已知值反复人为遮蔽 13 个样本，并按真实缺失的风格分布抽样，比较基线、Ridge、ElasticNet、PLS 和非负最小二乘。
- 所有标准化和模型调参都在训练折内部完成。最终数值矩阵只保留数值，回填来源和未检出标记另存。


## 数据变换


处理顺序：

1. 先根据文件 8 识别每列在文件 1 中是原值、自然对数还是常用对数。
2. 只对 51 个有标定浓度的化合物执行“逆变换 → 单位统一为 mg/L → log10”；其余变量不强行换算单位。
3. 将文件 8 为 0 的缺失位置标记为未检出，不直接当零值回填。
4. 对未检出位置建立三种左删失方案，主矩阵采用半最小正值方案，另外两种用于敏感性比较。
5. 对 `aroma_hops_sum` 进行按风格分层的重复遮蔽验证，再用验证误差最低的候选线性方法回填。
6. 输出数值矩阵、未检出掩码、敏感性记录、模型验证结果和回填日志；不把字符串来源列混入建模矩阵。


### 对数变换

In [3]:
def fix_mojibake(text):
    """修复 8.xlsx chemical_name 的编码乱码。"""
    text = str(text)
    try:
        return text.encode("latin-1").decode("utf-8")
    except (UnicodeDecodeError, UnicodeEncodeError):
        return text


# --- 0. 加载 ---
chem = pd.read_csv(RAW_DIR / "1.csv", encoding="utf-8-sig")
thresholds = pd.read_excel(RAW_DIR / "8.xlsx", sheet_name="Sensory thresholds")
column_meta = thresholds.drop_duplicates("field_name_R").set_index("field_name_R")
cal8 = pd.read_excel(RAW_DIR / "8.xlsx", sheet_name="Calibrated chemical dataset")

measurement_columns = chem.columns[3:]
chem_values = chem.set_index("beer")[measurement_columns].apply(pd.to_numeric, errors="coerce")
cb = chem.set_index("beer")
ab = cal8.set_index("beer").reindex(cb.index)

# --- 1. 用配对值识别文件 1 的实际尺度 ---
relation_rows = []
for col in (c for c in cal8.columns[3:] if c in measurement_columns):
    f1 = pd.to_numeric(cb[col], errors="coerce")
    f8 = pd.to_numeric(ab[col], errors="coerce")
    paired = f1.notna() & f8.notna()
    candidates = {
        "original": (paired, f8),
        "ln": (paired & (f8 > 0), np.log(f8.where(f8 > 0))),
        "log10": (paired & (f8 > 0), np.log10(f8.where(f8 > 0))),
    }
    support = {}
    support_n = {}
    for name, (mask, expected) in candidates.items():
        support_n[name] = int(mask.sum())
        support[name] = (float(np.isclose(f1[mask], expected[mask], rtol=1e-6, atol=1e-6).mean())
                         if mask.any() else np.nan)
    best = max(support, key=lambda k: -np.inf if pd.isna(support[k]) else support[k])
    relation_rows.append({"column": col, "selected": best, **{f"support_{k}": v for k, v in support.items()},
                          **{f"n_{k}": v for k, v in support_n.items()}})

relation_audit = pd.DataFrame(relation_rows).set_index("column")
relation = relation_audit["selected"]

STANDARD_16 = ["SO2.mg.L.", "acetic_acid.g.L.", "ammonia.mg.L.", "color.EBC.",
               "betaglucan.mg.L.", "lactic_acid.mg.L.", "glycerol", "iron", "pH",
               "protein.g.L.", "bitterness", "GFS", "ethanol..v.v.", "CO2.PSI.",
               "unfermentables", "kcalperc"]
FID_FPD_7 = ["acetaldehyde", "ethyl_acetate", "CS2", "DES", "DMS", "H2S", "MeSH"]
UNFLAGGED_3 = ["X3.methylbutanal", "ethyl.pentanoate", "X2.ethyl.3.methylpyrazine"]

cal_flag = column_meta["calibrated"].fillna("").reindex(measurement_columns)
is_sum = pd.Series(measurement_columns, index=measurement_columns).str.endswith("_sum")
floor_value = chem_values.min()
is_floor = floor_value.abs().between(1.8178185, 1.8178186)

block = pd.Series("GC-MS 相对丰度", index=measurement_columns)
block[is_sum] = "派生变量"
block[measurement_columns.isin(UNFLAGGED_3)] = "文件8无校准标记"
block[measurement_columns.isin(FID_FPD_7)] = "GC-FID/FPD 目标定量"
block[measurement_columns.isin(STANDARD_16)] = "标准理化指标"
block[cal_flag.str.contains("yes_gcms")] = "GC-MS 校准化合物"

# --- 2. 构造统一的建模尺度；零和缺失暂时保留为 NA ---
chem_prepared = chem_values.copy()
unit_of = column_meta["unit"].reindex(measurement_columns)
calibrated_51 = block.index[block.isin(["GC-MS 校准化合物", "GC-FID/FPD 目标定量"])]

for col in calibrated_51:
    f1 = chem_values[col]
    if relation[col] == "log10":
        concentration = 10 ** f1
    elif relation[col] == "ln":
        concentration = np.exp(f1)
    else:
        concentration = f1.copy()
    if unit_of[col] == "ug/L":
        concentration = concentration / 1000.0
    chem_prepared[col] = np.log10(concentration.where(concentration > 0))

feature_audit = pd.DataFrame({
    "block": block,
    "unit_in_file8": unit_of,
    "transform_relation": relation.reindex(measurement_columns),
    "scale_action": np.select(
        [block.eq("标准理化指标"),
         block.isin(["GC-MS 校准化合物", "GC-FID/FPD 目标定量"]),
         block.eq("派生变量")],
        ["保留文件1的log10(原单位)", "逆变换→统一mg/L→log10", "暂不变换；单独处理"],
        default="保留文件1的对数峰面积/A.U.尺度"),
    "has_shared_floor": is_floor,
    "missing_n_before_censoring": chem_values.isna().sum(),
})

display(relation_audit[["selected", "support_original", "support_ln", "support_log10"]]
        .groupby("selected").size().rename("columns"))
display(feature_audit.groupby("block").agg(columns=("block", "size"),
                                             shared_floor=("has_shared_floor", "sum"),
                                             missing=("missing_n_before_censoring", "sum")))

# 15 个无共同底限列分布
semi_cols = block.index[block.eq("GC-MS 相对丰度")]
no_floor_cols = semi_cols[~is_floor[semi_cols]]
no_floor_summary = chem_values[no_floor_cols].agg(["min", "median", "max", "skew"]).T
display(Markdown("**无共同底限列待核对**"))
display(no_floor_summary)


selected
log10       37
original    33
Name: columns, dtype: int64

,columns,shared_floor,missing
block,,,
GC-FID/FPD 目标定量,7,0,0
GC-MS 校准化合物,44,0,53
GC-MS 相对丰度,157,142,0
文件8无校准标记,3,0,7
标准理化指标,16,0,0
派生变量,4,0,13


**无共同底限列待核对**

,min,median,max,skew
n.pentanol,5.699395,7.981614,8.668641,-1.209874
X2.methyl.1.butanol,5.791555,8.180695,9.049597,-1.272326
isopentyl.formate,5.248896,8.156034,9.023886,-1.566796
X3.methyl.2.butenal,4.813145,6.001692,7.353396,0.258356
X3.methyl.2.hexanol,7.022097,7.752488,8.719027,0.224305
X3.furanmethanol,6.010021,6.623567,8.020248,0.829275
n.heptanol,4.838275,5.909912,7.248115,0.178399
ethyl.hex..3E..enoate,4.454240,6.130349,7.233225,-0.892491
n.octanol,4.757805,6.959388,7.798775,-1.799392
terpinolene,4.017765,6.132047,8.756311,0.369254


对 A.U 的处理

In [4]:
# --- A.U. 三列是否应取 log10:四条证据 ---
au_cols = list(block.index[block.eq("文件8无校准标记")])

# 证据1:file1 存的是 A.U. 原值,不是对数(file1→file8 配对支持度)
au_relation = relation_audit.loc[au_cols, ["selected", "support_original", "support_ln", "support_log10"]]

# 证据2-4:取log无损性、原尺度形态、log10后的形态
au_evidence_rows = []
for col in au_cols:
    raw = chem_values[col].dropna()
    log_values = np.log10(raw[raw > 0])
    au_evidence_rows.append({
        "column": col,
        "n": len(raw),
        "n_nonpositive": int((raw <= 0).sum()),
        "raw_skew": float(raw.skew()),
        "raw_decades": float(np.log10(raw.max() / raw.min())),
        "log_iqr": float(log_values.quantile(0.75) - log_values.quantile(0.25)),
        "log_skew": float(log_values.skew()),
    })
au_evidence = pd.DataFrame(au_evidence_rows).set_index("column")

# 对照包络:157个相对丰度列 + 16个标准理化列,本身已是log尺度
family_rows = []
for col in block.index[block.isin(["GC-MS 相对丰度", "标准理化指标"])]:
    v = chem_values[col].dropna()
    family_rows.append({"block": block[col],
                        "log_iqr": v.quantile(0.75) - v.quantile(0.25),
                        "log_skew": v.skew()})
family_envelope = pd.DataFrame(family_rows).groupby("block").agg(
    columns=("block", "size"),
    log_iqr_min=("log_iqr", "min"), log_iqr_median=("log_iqr", "median"), log_iqr_max=("log_iqr", "max"),
    log_skew_min=("log_skew", "min"), log_skew_max=("log_skew", "max"))

assert (au_evidence["n_nonpositive"] == 0).all(), "A.U. 列存在非正观测,不能无损取log"
display(au_relation)
display(au_evidence.round(3))
display(family_envelope.round(3))


,selected,support_original,support_ln,support_log10
column,,,,
X3.methylbutanal,original,0.995868,0.0,0.0
ethyl.pentanoate,original,0.984000,0.0,0.0
X2.ethyl.3.methylpyrazine,original,0.991968,0.0,0.0


,n,n_nonpositive,raw_skew,raw_decades,log_iqr,log_skew
column,,,,,,
X3.methylbutanal,243,0,11.673,10.364,0.788,-2.181
ethyl.pentanoate,250,0,9.236,2.589,0.455,1.208
X2.ethyl.3.methylpyrazine,250,0,6.040,4.413,0.817,0.584


,columns,log_iqr_min,log_iqr_median,log_iqr_max,log_skew_min,log_skew_max
block,,,,,,
GC-MS 相对丰度,157,0.000,0.940,8.781,-6.577,2.218
标准理化指标,16,0.018,0.282,1.250,-5.226,1.485


In [5]:
# --- 对3个未标定 A.U. 列取 log10:只改形态,不改变"半定量 A.U."语义 ---
# 有标定的51列走"逆变换→统一mg/L→log10";这三列无标定、无单位可逆推,
# 只做纯 log10。unit 仍按 file8 记为 A.U.,不解释为浓度。
for col in au_cols:
    chem_prepared[col] = np.log10(chem_values[col].where(chem_values[col] > 0))

feature_audit.loc[au_cols, "scale_action"] = "log10(A.U.原值)"

# 校验:NA 数不变;变换后取值进入对数量级
au_transform_check = pd.DataFrame({
    "n_na_before": chem_values[au_cols].isna().sum(),
    "n_na_after": chem_prepared[au_cols].isna().sum(),
    "min_after": chem_prepared[au_cols].min(),
    "max_after": chem_prepared[au_cols].max(),
})
assert (au_transform_check["n_na_before"] == au_transform_check["n_na_after"]).all()
display(au_transform_check.round(4))
display(feature_audit.groupby(["block", "scale_action"]).size().rename("columns").to_frame())


,n_na_before,n_na_after,min_after,max_after
X3.methylbutanal,7,7,-8.2596,2.1040
ethyl.pentanoate,0,0,-2.3645,0.2250
X2.ethyl.3.methylpyrazine,0,0,-4.3565,0.0563


,,columns
block,scale_action,
GC-FID/FPD 目标定量,逆变换→统一mg/L→log10,7
GC-MS 校准化合物,逆变换→统一mg/L→log10,44
GC-MS 相对丰度,保留文件1的对数峰面积/A.U.尺度,157
文件8无校准标记,log10(A.U.原值),3
标准理化指标,保留文件1的log10(原单位),16
派生变量,暂不变换；单独处理,4


### 缺失值
60 NA (nondetect, file8=0) spread across 8 columns: 7 in 文件8无校准标记 (X3.methylbutanal), 53 in GC-MS 校准化合物 (X2.3.butanedione 9, amyl.acetate 27, ethyl.isobutyrate 1, ethyl.nicotinate 2, furfural 1, methylpyrazine 12, phenylacetaldehyde 1)

13 NA in 派生变量 (aroma_hops_sum)

Blocks with zero missing: 标准理化指标 (0), GC-FID/FPD (0), GC-MS 相对丰度 (0 — their non-detects are written as the shared floor value −1.8178, not NA)

In [6]:
# --- 缺失机制识别：缺失、未检出和派生变量缺失分开 ---
na_mask = chem_values.isna()
nd_mask = pd.DataFrame(False, index=chem_values.index, columns=chem_values.columns)
missing_rows = []

for col in na_mask.columns[na_mask.any()]:
    for beer in chem_values.index[na_mask[col]]:
        if col == "aroma_hops_sum":
            kind, value8 = "derived_missing", np.nan
        elif col in ab.columns:
            value8 = pd.to_numeric(pd.Series([ab.loc[beer, col]]), errors="coerce").iloc[0]
            if pd.isna(value8):
                kind = "source_missing"
            elif value8 == 0:
                kind = "nondetect_left_censored"
                nd_mask.loc[beer, col] = True
            else:
                kind = "file8_positive_measurement"
        else:
            kind, value8 = "no_file8_counterpart", np.nan
        missing_rows.append({"beer": beer, "column": col, "file8_value": value8, "missing_type": kind})

missing_audit = pd.DataFrame(missing_rows)
display(missing_audit.groupby(["missing_type", "column"], dropna=False).size().rename("n").to_frame())

unexpected = missing_audit[~missing_audit["missing_type"].isin(
    ["nondetect_left_censored", "derived_missing"])]
assert unexpected.empty, f"发现未纳入规则的缺失值：\n{unexpected}"
assert int(nd_mask.to_numpy().sum()) == 60
assert int((missing_audit["missing_type"] == "derived_missing").sum()) == 13


n
missing_type            column                
derived_missing         aroma_hops_sum      13
nondetect_left_censored X2.3.butanedione     9
                        X3.methylbutanal     7
                        amyl.acetate        27
                        ethyl.isobutyrate    1
                        ethyl.nicotinate     2
                        furfural             1
                        methylpyrazine      12
                        phenylacetaldehyde   1

NA集中在 8 个没有共享底限的列(7 个校准列 + X3.methylbutanal)；文件1 NA 且文件8同位置为 0 → 均视为未检出

设定参照值 = 该列在建模尺度上的最小已观测值（>=5 obs），按min/2处理为底限值（half_min = observed.min() − log10(2)）

In [7]:
# --- 左删失敏感性方案 --- 
from scipy.stats import truncnorm

rng = np.random.default_rng(20261025)
scenario_matrices = {name: chem_prepared.copy() for name in
                     ["half_min", "lod_sqrt2_proxy", "qrilc"]}
censor_rows = []

for col in nd_mask.columns[nd_mask.any()]:
    target = nd_mask.index[nd_mask[col]]
    observed = chem_prepared.loc[~nd_mask[col], col].dropna()
    assert len(observed) >= 5, f"{col} 可用于估计左尾的观测过少"

    # 全部列已在 log10 尺度:原尺度的"除以2"等价于减 log10(2)
    floor_half = observed.min() - np.log10(2.0)
    floor_sqrt2 = observed.min() - np.log10(np.sqrt(2.0))
    fit_values = observed

    # QRILC 风格的左尾随机填补；固定随机种子保证可复现
    mu = float(fit_values.median())
    mad = float(np.median(np.abs(fit_values - mu)))
    sigma = max(1.4826 * mad, float(fit_values.std(ddof=1)) * 0.25, 1e-6)
    upper = float(fit_values.min())
    qrilc_fit = truncnorm.rvs(-np.inf, (upper - mu) / sigma,
                              loc=mu, scale=sigma, size=len(target), random_state=rng)
    qrilc_values = qrilc_fit

    scenario_matrices["half_min"].loc[target, col] = floor_half
    scenario_matrices["lod_sqrt2_proxy"].loc[target, col] = floor_sqrt2
    scenario_matrices["qrilc"].loc[target, col] = qrilc_values

    for beer, qvalue in zip(target, qrilc_values):
        censor_rows.extend([
            {"beer": beer, "column": col, "method": "half_min", "filled_value": floor_half},
            {"beer": beer, "column": col, "method": "lod_sqrt2_proxy", "filled_value": floor_sqrt2},
            {"beer": beer, "column": col, "method": "qrilc", "filled_value": qvalue},
        ])

censor_sensitivity = pd.DataFrame(censor_rows)
display(censor_sensitivity.groupby(["column", "method"])["filled_value"]
        .agg(["count", "min", "median", "max"]))

# 主矩阵采用与数据集现有底限规则一致的 half_min；其他方案只用于敏感性分析
chem_model = scenario_matrices["half_min"].copy()
assert not chem_model.drop(columns=["aroma_hops_sum"]).isna().any().any()


count       min    median       max
column             method                                              
X2.3.butanedione   half_min             9 -7.530178 -7.530178 -7.530178
                   lod_sqrt2_proxy      9 -7.379663 -7.379663 -7.379663
                   qrilc                9 -7.418008 -7.315618 -7.247016
X3.methylbutanal   half_min             7 -8.560667 -8.560667 -8.560667
                   lod_sqrt2_proxy      7 -8.410152 -8.410152 -8.410152
                   qrilc                7 -8.406675 -8.279425 -8.262200
amyl.acetate       half_min            27 -9.301030 -9.301030 -9.301030
                   lod_sqrt2_proxy     27 -9.150515 -9.150515 -9.150515
                   qrilc               27 -9.839357 -9.157309 -9.016845
ethyl.isobutyrate  half_min             1 -9.602060 -9.602060 -9.602060
                   lod_sqrt2_proxy      1 -9.451545 -9.451545 -9.451545
                   qrilc                1 -9.402619 -9.402619 -9.402619
ethyl.nicotinate   half_min             2 -6.920819 -6.920819 -6.920819
                   lod_sqrt2_proxy      2 -6.770304 -6.770304 -6.770304
                   qrilc                2 -6.729830 -6.697706 -6.665581
furfural           half_min             1 -7.491470 -7.491470 -7.491470
                   lod_sqrt2_proxy      1 -7.340955 -7.340955 -7.340955
                   qrilc                1 -7.196988 -7.196988 -7.196988
methylpyrazine     half_min            12 -7.920819 -7.920819 -7.920819
                   lod_sqrt2_proxy     12 -7.770304 -7.770304 -7.770304
                   qrilc               12 -8.498941 -7.826141 -7.632537
phenylacetaldehyde half_min             1 -6.915424 -6.915424 -6.915424
                   lod_sqrt2_proxy      1 -6.764909 -6.764909 -6.764909
                   qrilc                1 -6.623281 -6.623281 -6.623281

aroma_hops_sum：使用组分拟合/验证

特征:从阈值表 origin_paper == "Hops" 取 hops 组分作 X
验证设计:100 次重复,每次按真实 13 个缺失值的 tasting_category_fine 风格构成,从已知样本分层抽 13 个做验证集(模拟真实缺失分布)
候选:RidgeCV、ElasticNetCV、PLS、NNLS + 全局/风格中位数

In [8]:
from sklearn.base import clone
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import ElasticNetCV, LinearRegression, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [9]:
# ===== aroma_hops_sum：按真实缺失风格分布做重复遮蔽 =====
style = chem.set_index("beer")["tasting_category_fine"].reindex(chem_model.index)
th = thresholds.drop_duplicates("field_name_R")
hops = [c for c in th.loc[th["origin_paper"].eq("Hops"), "field_name_R"]
        if c in chem_model.columns and not c.endswith("_sum")]

y_all = chem_model["aroma_hops_sum"]
X_all = chem_model[hops]
observed_idx = y_all.index[y_all.notna() & X_all.notna().all(axis=1)]
missing_idx = y_all.index[y_all.isna()]
missing_style_counts = style.loc[missing_idx].value_counts(dropna=False)

models = {
    "ridge": Pipeline([
        ("scale", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-3, 3, 25), cv=5)),
    ]),
    "elastic_net": Pipeline([
        ("scale", StandardScaler()),
        ("model", ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9, 1.0], alphas=np.logspace(-3, 1, 30),
                                cv=5, max_iter=20000, random_state=0)),
    ]),
    "pls": Pipeline([
        ("scale", StandardScaler()),
        ("model", PLSRegression(n_components=min(5, len(hops)), scale=False)),
    ]),
    "nnls": Pipeline([
        ("scale", StandardScaler()),
        ("model", LinearRegression(positive=True)),
    ]),
}


def stratified_mask_indices(rng_local):
    """按13个真实缺失值的风格构成，从已知样本中抽取验证集。"""
    chosen = []
    for style_name, n_needed in missing_style_counts.items():
        pool = observed_idx[style.loc[observed_idx].eq(style_name)]
        n_take = min(int(n_needed), len(pool))
        if n_take:
            chosen.extend(rng_local.choice(pool.to_numpy(), size=n_take, replace=False).tolist())
    if len(chosen) < len(missing_idx):
        remaining = observed_idx.difference(pd.Index(chosen))
        chosen.extend(rng_local.choice(remaining.to_numpy(),
                                       size=len(missing_idx) - len(chosen), replace=False).tolist())
    return pd.Index(chosen)


rng_mask = np.random.default_rng(20261025)
validation_rows = []
for repeat in range(100):
    valid_idx = stratified_mask_indices(rng_mask)
    train_idx = observed_idx.difference(valid_idx)
    X_train, y_train = X_all.loc[train_idx], y_all.loc[train_idx]
    X_valid, y_valid = X_all.loc[valid_idx], y_all.loc[valid_idx]

    baseline_predictions = {
        "global_median": np.repeat(y_train.median(), len(valid_idx)),
        "style_median": style.loc[valid_idx].map(y_train.groupby(style.loc[train_idx]).median())
                          .fillna(y_train.median()).to_numpy(),
    }
    predictions = dict(baseline_predictions)
    for name, estimator in models.items():
        fitted = clone(estimator).fit(X_train, y_train)
        predictions[name] = np.asarray(fitted.predict(X_valid)).reshape(-1)

    for name, pred in predictions.items():
        validation_rows.append({
            "repeat": repeat,
            "method": name,
            "n_valid": len(valid_idx),
            "MAE": mean_absolute_error(y_valid, pred),
            "RMSE": mean_squared_error(y_valid, pred) ** 0.5,
            "R2": r2_score(y_valid, pred),
        })

aroma_validation = pd.DataFrame(validation_rows)
validation_summary = aroma_validation.groupby("method").agg(
    repeats=("repeat", "nunique"),
    MAE_mean=("MAE", "mean"),
    MAE_sd=("MAE", "std"),
    RMSE_mean=("RMSE", "mean"),
    RMSE_sd=("RMSE", "std"),
    R2_mean=("R2", "mean"),
).sort_values("RMSE_mean")

display(Markdown(f"**Hops 候选组分 {len(hops)} 列；已知目标 {len(observed_idx)}；待填补 {len(missing_idx)}。**"))
display(validation_summary)


**Hops 候选组分 77 列；已知目标 237；待填补 13。**

,repeats,MAE_mean,MAE_sd,RMSE_mean,RMSE_sd,R2_mean
method,,,,,,
nnls,100,0.046883,0.008664,0.057843,0.011427,0.989574
elastic_net,100,0.048655,0.009912,0.061776,0.013212,0.988124
ridge,100,0.068220,0.012333,0.085711,0.015648,0.977572
pls,100,0.126143,0.027884,0.152644,0.033036,0.927748
style_median,100,0.451723,0.075353,0.536725,0.094553,0.092054
global_median,100,0.633870,0.097145,0.791322,0.089362,-0.844015


In [10]:
# 只在线性/降维候选方法中选最终模型；中位数作为性能基线，不作为首选公式
candidate_methods = list(models)
selected_method = validation_summary.loc[candidate_methods, "RMSE_mean"].idxmin()
selected_model = clone(models[selected_method]).fit(X_all.loc[observed_idx], y_all.loc[observed_idx])
aroma_prediction = pd.Series(
    np.asarray(selected_model.predict(X_all.loc[missing_idx])).reshape(-1),
    index=missing_idx,
    name="filled_value",
)

aroma_fill_log = pd.DataFrame({
    "beer": missing_idx,
    "column": "aroma_hops_sum",
    "filled_value": aroma_prediction.to_numpy(),
    "method": selected_method,
    "validation_RMSE": validation_summary.loc[selected_method, "RMSE_mean"],
    "validation_MAE": validation_summary.loc[selected_method, "MAE_mean"],
})

display(Markdown(
    f"**最终选择 `{selected_method}`：重复分层遮蔽 RMSE="
    f"{validation_summary.loc[selected_method, 'RMSE_mean']:.4f}，"
    f"MAE={validation_summary.loc[selected_method, 'MAE_mean']:.4f}。**"))
display(aroma_fill_log)


**最终选择 `nnls`：重复分层遮蔽 RMSE=0.0578，MAE=0.0469。**

,beer,column,filled_value,method,validation_RMSE,validation_MAE
0,64,aroma_hops_sum,2.231947,nnls,0.057843,0.046883
1,158,aroma_hops_sum,2.022307,nnls,0.057843,0.046883
2,224,aroma_hops_sum,3.351997,nnls,0.057843,0.046883
3,172,aroma_hops_sum,3.392197,nnls,0.057843,0.046883
4,7,aroma_hops_sum,2.639602,nnls,0.057843,0.046883
5,151,aroma_hops_sum,3.154483,nnls,0.057843,0.046883
6,219,aroma_hops_sum,3.537538,nnls,0.057843,0.046883
7,39,aroma_hops_sum,3.595511,nnls,0.057843,0.046883
8,80,aroma_hops_sum,3.136708,nnls,0.057843,0.046883
9,120,aroma_hops_sum,3.100354,nnls,0.057843,0.046883


In [11]:
# ===== 输出：数值、标记和验证记录分开保存 =====
final_matrix = chem_model.copy()
final_matrix.loc[missing_idx, "aroma_hops_sum"] = aroma_prediction
assert final_matrix.shape == chem_values.shape
assert not final_matrix.isna().any().any()
assert all(pd.api.types.is_numeric_dtype(final_matrix[c]) for c in final_matrix.columns)

feature_audit.to_csv(INTERIM_DIR / "chem_feature_provenance_v1.csv", encoding="utf-8-sig")
relation_audit.to_csv(INTERIM_DIR / "file1_file8_transform_support_v1.csv", encoding="utf-8-sig")
missing_audit.to_csv(INTERIM_DIR / "chem_missing_mechanism_v1.csv", index=False, encoding="utf-8-sig")
nd_mask.astype("uint8").to_csv(INTERIM_DIR / "chem_nondetect_mask_v1.csv")
censor_sensitivity.to_csv(INTERIM_DIR / "chem_nondetect_sensitivity_v1.csv", index=False)
aroma_validation.to_csv(INTERIM_DIR / "aroma_hops_sum_masked_validation_v1.csv", index=False)
aroma_fill_log.to_csv(INTERIM_DIR / "aroma_hops_sum_fill_log_v1.csv", index=False)

# 主矩阵本身就是 half_min 情景；只额外保存两个替代情景，避免字节重复
for method in ["lod_sqrt2_proxy", "qrilc"]:
    matrix = scenario_matrices[method]
    output = matrix.copy()
    output.loc[missing_idx, "aroma_hops_sum"] = aroma_prediction
    output.to_csv(INTERIM_DIR / f"chem_log10_matrix_{method}_v1.csv")

v1_path = INTERIM_DIR / "chem_log10_matrix_v1.csv"
final_matrix.to_csv(v1_path)
v1_manifest = {
    "version": "v1",
    "rows": int(final_matrix.shape[0]),
    "chemical_features": int(final_matrix.shape[1]),
    "nondetect_rule": "half_min on the variable's retained modeling scale",
    "aroma_hops_sum_method": selected_method,
    "aroma_validation_repeats": 100,
    "sha256": hashlib.sha256(v1_path.read_bytes()).hexdigest(),
    "next_stage_input": "chem_log10_matrix_v1.csv",
}
(INTERIM_DIR / "chem_log10_matrix_v1_manifest.json").write_text(
    json.dumps(v1_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("固定主矩阵: chem_log10_matrix_v1.csv")
print("敏感性情景: half_min=v1 / lod_sqrt2_proxy / qrilc")
print(f"aroma_hops_sum 最终方法: {selected_method}")


固定主矩阵: chem_log10_matrix_v1.csv
敏感性情景: half_min=v1 / lod_sqrt2_proxy / qrilc
aroma_hops_sum 最终方法: nnls


In [12]:
# 最终检查：主矩阵不能混入来源字符串；所有回填均可由日志定位
output_checks = pd.Series({
    "rows": final_matrix.shape[0],
    "chemical_features": final_matrix.shape[1],
    "remaining_NA": int(final_matrix.isna().sum().sum()),
    "nondetect_cells": int(nd_mask.to_numpy().sum()),
    "aroma_hops_sum_imputed": len(aroma_fill_log),
    "selected_aroma_method": selected_method,
})
display(output_checks.to_frame("value"))


,value
rows,250
chemical_features,231
remaining_NA,0
nondetect_cells,60
aroma_hops_sum_imputed,13
selected_aroma_method,nnls


## 3. 异常值标记与展示

伪影 (artifact) = 标记本身反映的不是"这个样品异常",而是"分析方法在这个数据形态下失灵"——量纲/尺度的副作用,不是数据里的真信号。

异常值检测分为两个层次：

1. **单变量异常单元格**：同时计算风格内和全体样本的 median/MAD 稳健 Z 分数，二者均满足 `|z| > 4.5` 才标记。风格内有效样本少于 8 时不作单变量判断，防止样本量太小导致 MAD 失效；共享检测底限、未检出填补值及 `aroma_hops_sum` 模型填充值不参与判断。
2. **多变量异常样本**：对非派生化学变量先减去各风格中位数，再进行稳健缩放。PCA 得分空间使用最小协方差行列式（MCD）马氏距离，残差空间使用正交距离；Isolation Forest 作为独立的非线性交叉证据。

`review_flag` 表示至少一种多变量方法提示复核，`high_confidence_flag` 表示 PCA 结构异常与 Isolation Forest 同时提示。


数据读取：固定版本 chem_log10_matrix_v1.csv

In [13]:
from sklearn.covariance import MinCovDet
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

# 后续异常值分析只读取固定版本，避免使用内存中的临时情景矩阵
outlier_data = pd.read_csv(INTERIM_DIR / "chem_log10_matrix_v1.csv", index_col=0)
outlier_data.index = outlier_data.index.astype(chem_values.index.dtype)
style_v1 = chem.set_index("beer")["tasting_category_fine"].reindex(outlier_data.index)
nd_mask_v1 = pd.read_csv(INTERIM_DIR / "chem_nondetect_mask_v1.csv", index_col=0).astype(bool)
nd_mask_v1.index = nd_mask_v1.index.astype(outlier_data.index.dtype)

imputed_mask = nd_mask_v1.reindex_like(outlier_data).fillna(False)
imputed_mask.loc[aroma_fill_log["beer"], "aroma_hops_sum"] = True


def robust_location_scale(values):
    values = pd.Series(values).dropna().astype(float)
    center = float(values.median())
    mad_scale = 1.4826 * float(np.median(np.abs(values - center)))
    if not np.isfinite(mad_scale) or mad_scale <= 1e-12:
        q25, q75 = values.quantile([0.25, 0.75])
        mad_scale = float((q75 - q25) / 1.349)
    return center, mad_scale


# --- 单变量：风格内稳健 Z 分数 ---
cell_flag_rows = []
for col in feature_audit.index[~feature_audit["block"].eq("派生变量")]:
    eligible = ~imputed_mask[col]
    if bool(feature_audit.loc[col, "has_shared_floor"]):
        # 共同底限是左删失结构，不是异常低值
        eligible &= ~np.isclose(outlier_data[col], outlier_data[col].min(), atol=1e-8)
    global_valid = outlier_data.loc[eligible, col]
    global_center, global_scale = robust_location_scale(global_valid)
    for style_name, idx in style_v1.groupby(style_v1, dropna=False).groups.items():
        idx = pd.Index(idx)
        valid_idx = idx[eligible.loc[idx]]
        if len(valid_idx) < 8:
            continue
        center, scale = robust_location_scale(outlier_data.loc[valid_idx, col])
        if (not np.isfinite(scale) or scale <= 1e-12 or
                not np.isfinite(global_scale) or global_scale <= 1e-12):
            continue
        z = (outlier_data.loc[valid_idx, col] - center) / scale
        z_global = (outlier_data.loc[valid_idx, col] - global_center) / global_scale
        flagged = z.index[(z.abs() > 4.5) & (z_global.abs() > 4.5) &
                          (np.sign(z) == np.sign(z_global))]
        for beer in flagged:
            cell_flag_rows.append({
                "beer": beer,
                "style": style_name,
                "column": col,
                "block": feature_audit.loc[col, "block"],
                "value": outlier_data.loc[beer, col],
                "reference": "within_style_and_global",
                "robust_z": z.loc[beer],
                "global_robust_z": z_global.loc[beer],
                "direction": "high" if z.loc[beer] > 0 else "low",
                "imputed": False,
            })

cell_outlier_flags = pd.DataFrame(cell_flag_rows)
display(Markdown(f"**单变量候选异常：{len(cell_outlier_flags)} 个单元格，涉及 "
                 f"{cell_outlier_flags['beer'].nunique() if len(cell_outlier_flags) else 0} 个样本。**"))
display(cell_outlier_flags.reindex(cell_outlier_flags["robust_z"].abs().sort_values(ascending=False).index).head(25))


**单变量候选异常：146 个单元格，涉及 77 个样本。**

,beer,style,column,block,value,reference,robust_z,global_robust_z,direction,imputed
143,177,Fruitbeer,ethyl.pentanoate,文件8无校准标记,0.225018,within_style_and_global,62.640332,5.364202,high,False
25,40,Low/No alcohol,kcalperc,标准理化指标,0.955286,within_style_and_global,-28.887812,-7.015172,low,False
23,96,Pils/Lager,unfermentables,标准理化指标,0.390935,within_style_and_global,-20.894327,-6.792385,low,False
98,158,Amber,eugenol,GC-MS 校准化合物,-0.366713,within_style_and_global,20.607663,6.344879,high,False
14,119,Pils/Lager,isobutanol,GC-MS 校准化合物,-1.321118,within_style_and_global,-19.614528,-11.279272,low,False
112,67,Stout/Porter,gamma.nonalactone,GC-MS 校准化合物,3.358688,within_style_and_global,19.534351,12.182004,high,False
99,32,Amber,eugenol,GC-MS 校准化合物,-0.505181,within_style_and_global,19.415195,5.988342,high,False
29,221,Dubbel,X2.3.butanedione,GC-MS 校准化合物,-5.200659,within_style_and_global,-19.203777,-6.473992,low,False
113,103,Stout/Porter,gamma.nonalactone,GC-MS 校准化合物,-3.638879,within_style_and_global,-18.178649,-10.677997,low,False
11,175,Strong ale,isoamyl_alcohol,GC-MS 校准化合物,-0.730837,within_style_and_global,-17.505863,-13.968922,low,False


In [14]:
# --- 多变量：风格中心化 + 稳健缩放 + PCA/MCD + Isolation Forest ---
multivariate_cols = feature_audit.index[~feature_audit["block"].eq("派生变量")]
X_mv = outlier_data[multivariate_cols].copy()

# 减去风格中位数，避免正常的风格化学差异被误判为异常
style_medians = X_mv.groupby(style_v1).transform("median")
X_centered = X_mv - style_medians
scaler_mv = RobustScaler(quantile_range=(25, 75))
X_scaled = scaler_mv.fit_transform(X_centered)

# 最多保留10个主成分，使MCD在250个样本下保持可估计性
pca_full = PCA().fit(X_scaled)
n80 = int(np.searchsorted(np.cumsum(pca_full.explained_variance_ratio_), 0.80) + 1)
n_components = max(2, min(10, n80))
pca_outlier = PCA(n_components=n_components).fit(X_scaled)
scores = pca_outlier.transform(X_scaled)
reconstruction = pca_outlier.inverse_transform(scores)
orthogonal_distance = np.sqrt(np.square(X_scaled - reconstruction).sum(axis=1))

mcd = MinCovDet(support_fraction=0.75, random_state=20261025).fit(scores)
score_distance_sq = mcd.mahalanobis(scores)
score_cutoff_theoretical = stats.chi2.ppf(0.99, df=n_components)
score_cutoff = max(float(np.quantile(score_distance_sq, 0.99)), score_cutoff_theoretical)
orthogonal_cutoff = float(np.quantile(orthogonal_distance, 0.99))
pca_flag = (score_distance_sq > score_cutoff) | (orthogonal_distance > orthogonal_cutoff)

iso = IsolationForest(n_estimators=1000, contamination="auto", random_state=20261025)
iso.fit(X_scaled)
isolation_score = iso.score_samples(X_scaled)
isolation_cutoff = float(np.quantile(isolation_score, 0.01))
isolation_flag = isolation_score <= isolation_cutoff

sample_outlier_scores = pd.DataFrame({
    "beer": outlier_data.index,
    "style": style_v1.to_numpy(),
    "score_distance_sq": score_distance_sq,
    "score_cutoff_99": score_cutoff,
    "orthogonal_distance": orthogonal_distance,
    "orthogonal_cutoff_99": orthogonal_cutoff,
    "pca_structure_flag": pca_flag,
    "isolation_score": isolation_score,
    "isolation_cutoff_1pct": isolation_cutoff,
    "isolation_flag": isolation_flag,
    "review_flag": pca_flag | isolation_flag,
    "high_confidence_flag": pca_flag & isolation_flag,
    "univariate_flag_count": pd.Series(
        cell_outlier_flags["beer"].value_counts() if len(cell_outlier_flags) else {},
        dtype=float).reindex(outlier_data.index).fillna(0).astype(int).to_numpy(),
    "imputed_cell_count": imputed_mask.sum(axis=1).to_numpy(),
}).set_index("beer")

display(Markdown(
    f"**PCA 使用 {n_components} 个成分（累计解释 "
    f"{pca_outlier.explained_variance_ratio_.sum():.1%}）；"
    f"复核样本 {int(sample_outlier_scores['review_flag'].sum())} 个，"
    f"高置信样本 {int(sample_outlier_scores['high_confidence_flag'].sum())} 个。**"))
display(sample_outlier_scores.sort_values(
    ["high_confidence_flag", "review_flag", "univariate_flag_count", "score_distance_sq"],
    ascending=False).head(25))


**PCA 使用 10 个成分（累计解释 60.1%）；复核样本 7 个，高置信样本 2 个。**

,style,score_distance_sq,score_cutoff_99,orthogonal_distance,orthogonal_cutoff_99,pca_structure_flag,isolation_score,isolation_cutoff_1pct,isolation_flag,review_flag,high_confidence_flag,univariate_flag_count,imputed_cell_count
beer,,,,,,,,,,,,,
157,Tripel,751.425377,615.701117,30.623839,39.669896,True,-0.610388,-0.600176,True,True,True,6,0
103,Stout/Porter,687.656997,615.701117,36.567869,39.669896,True,-0.624494,-0.600176,True,True,True,5,1
234,Blond,624.596901,615.701117,33.870535,39.669896,True,-0.571677,-0.600176,False,True,False,4,0
192,Brut,536.200383,615.701117,39.225371,39.669896,False,-0.602066,-0.600176,True,True,False,0,0
168,Scotch,58.759887,615.701117,40.039638,39.669896,True,-0.448299,-0.600176,False,True,False,0,0
10,Low/No alcohol,51.118834,615.701117,45.237896,39.669896,True,-0.522828,-0.600176,False,True,False,0,2
223,Low/No alcohol,16.411439,615.701117,44.102506,39.669896,True,-0.476510,-0.600176,False,True,False,0,1
9,Hoppy,15.697543,615.701117,32.812443,39.669896,False,-0.452575,-0.600176,False,False,False,9,0
34,Tripel,17.844479,615.701117,37.658902,39.669896,False,-0.457145,-0.600176,False,False,False,8,0


In [15]:
# === 展示与落盘：只保存标记，不修改 v1 ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

if len(cell_outlier_flags):
    block_counts = cell_outlier_flags.groupby("block").size().sort_values()
    block_counts.plot.barh(ax=axes[0], color="#4C78A8")
axes[0].set_title("单变量异常标记（按变量类型）")
axes[0].set_xlabel("标记单元格数")
axes[0].set_ylabel("")

plot_flag = sample_outlier_scores["review_flag"].to_numpy()
axes[1].scatter(scores[~plot_flag, 0], scores[~plot_flag, 1],
                s=24, alpha=0.55, color="#9AA0A6", label="未标记")
axes[1].scatter(scores[plot_flag, 0], scores[plot_flag, 1],
                s=48, alpha=0.9, color="#D62728", label="需要复核")
for beer, x, y in zip(outlier_data.index[plot_flag], scores[plot_flag, 0], scores[plot_flag, 1]):
    axes[1].annotate(str(beer), (x, y), xytext=(3, 3), textcoords="offset points", fontsize=8)
axes[1].set_title("风格中心化后的 PCA 异常展示")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].legend(frameon=False)

plt.tight_layout()
outlier_figure = FIGURE_DIR / "02_outlier_overview_v1.png"
fig.savefig(outlier_figure, dpi=180, bbox_inches="tight")
plt.show()

cell_outlier_flags.to_csv(INTERIM_DIR / "chem_outlier_cell_flags_v1.csv", index=False, encoding="utf-8-sig")
sample_outlier_scores.to_csv(INTERIM_DIR / "chem_outlier_sample_scores_v1.csv", encoding="utf-8-sig")

assert hashlib.sha256((INTERIM_DIR / "chem_log10_matrix_v1.csv").read_bytes()).hexdigest() == v1_manifest["sha256"]
print("v1 数值矩阵未改变。异常标记表和展示图已保存。")


v1 数值矩阵未改变。异常标记表和展示图已保存。


### 3.1 高维距离的交叉验证

由于化学特征数（231）接近样本数（250），不直接使用样本协方差的普通马氏距离，也不依赖原始高维欧氏距离。增加以下互补证据：

|family	|成员方法	|衡量什么|
|---|---|---|
|global_family	|收缩马氏距离 + Ledoit–Wolf 协方差收缩	|全局椭球结构:离全体中心有多远|
|pca_family	|第 2 层的 pca_structure_flag(MCD 得分距离 + 正交距离)	|PCA 结构空间里的异常|
|local_family	|kNN 距离 + LOF	| 局部邻域和局部密度 |
|profile_family	| 相关距离 + 中心化对数谱距离	|谱形:化学成分的形状异常(对整体水平平移不敏感),仅对157个 GC-MS 相对丰度变量，在现有对数尺度上减去每个样本的均值后计算近邻距离|

每个方法保留连续得分和百分位。候选阈值统一取经验分布的97.5%尾部；同一family中的 kNN/LOF 或相关/对数谱只计一票，防止相似方法重复加权。


In [16]:
from sklearn.covariance import LedoitWolf
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors


def upper_percentile(values):
    """较大值更异常时的经验百分位。"""
    return pd.Series(values).rank(method="average", pct=True).to_numpy()


# 1) 全局：Ledoit–Wolf 收缩马氏距离
lw = LedoitWolf().fit(X_scaled)
lw_centered = X_scaled - lw.location_
shrinkage_md2 = np.einsum("ij,jk,ik->i", lw_centered, lw.precision_, lw_centered)

# 2) 局部：在白化 PCA 空间计算 kNN 和 LOF，减少距离集中
n_local_pc = min(30, X_scaled.shape[0] - 2, X_scaled.shape[1])
local_scores = PCA(n_components=n_local_pc, whiten=True, random_state=20261025).fit_transform(X_scaled)
k_neighbors = 15
neighbor_dist, _ = NearestNeighbors(n_neighbors=k_neighbors + 1).fit(local_scores).kneighbors(local_scores)
knn_distance = neighbor_dist[:, -1]
lof_model = LocalOutlierFactor(n_neighbors=20, contamination="auto")
lof_model.fit_predict(local_scores)
lof_score = -lof_model.negative_outlier_factor_  # 越大越异常

# 3) 谱形：相关距离的第15近邻
corr_dist, _ = NearestNeighbors(n_neighbors=k_neighbors + 1, metric="correlation").fit(X_scaled).kneighbors(X_scaled)
correlation_knn_distance = corr_dist[:, -1]

# 4) GC-MS 相对丰度块的中心化对数谱距离（Aitchison-like 敏感性证据）
relative_cols = feature_audit.index[feature_audit["block"].eq("GC-MS 相对丰度")]
log_profile = outlier_data[relative_cols].to_numpy(dtype=float)
centered_log_profile = log_profile - log_profile.mean(axis=1, keepdims=True)
profile_dist, _ = NearestNeighbors(n_neighbors=k_neighbors + 1).fit(centered_log_profile).kneighbors(centered_log_profile)
logratio_knn_distance = profile_dist[:, -1] / np.sqrt(len(relative_cols))

distance_scores = pd.DataFrame({
    "beer": outlier_data.index,
    "style": style_v1.to_numpy(),
    "shrinkage_md2": shrinkage_md2,
    "knn_distance": knn_distance,
    "lof_score": lof_score,
    "correlation_knn_distance": correlation_knn_distance,
    "logratio_knn_distance": logratio_knn_distance,
}).set_index("beer")

score_columns = ["shrinkage_md2", "knn_distance", "lof_score",
                 "correlation_knn_distance", "logratio_knn_distance"]
for col in score_columns:
    distance_scores[f"{col}_percentile"] = upper_percentile(distance_scores[col])
    distance_scores[f"{col}_flag"] = distance_scores[f"{col}_percentile"] >= 0.975

# 相近方法归为一个证据家族，避免重复计票
distance_scores["global_family"] = distance_scores["shrinkage_md2_flag"]
distance_scores["pca_family"] = sample_outlier_scores["pca_structure_flag"]
distance_scores["local_family"] = distance_scores[["knn_distance_flag", "lof_score_flag"]].any(axis=1)
distance_scores["profile_family"] = distance_scores[["correlation_knn_distance_flag",
                                                       "logratio_knn_distance_flag"]].any(axis=1)
family_cols = ["global_family", "pca_family", "local_family", "profile_family"]
distance_scores["evidence_families"] = distance_scores[family_cols].sum(axis=1)
distance_scores["distance_review_flag"] = distance_scores["evidence_families"] >= 1
distance_scores["distance_consensus_flag"] = distance_scores["evidence_families"] >= 2

distance_review = distance_scores.loc[distance_scores["distance_review_flag"]].sort_values(
    ["evidence_families", "shrinkage_md2_percentile"], ascending=False)
display(Markdown(
    f"**至少一种距离family标记 {len(distance_review)} 个样本；"
    f"至少两种独立family一致标记 {int(distance_scores['distance_consensus_flag'].sum())} 个样本。**"))
display(distance_review[["style", "evidence_families", *family_cols,
                         *[f"{c}_percentile" for c in score_columns]]])


**至少一种距离family标记 15 个样本；至少两种独立family一致标记 8 个样本。**

,style,evidence_families,global_family,pca_family,local_family,profile_family,shrinkage_md2_percentile,knn_distance_percentile,lof_score_percentile,correlation_knn_distance_percentile,logratio_knn_distance_percentile
beer,,,,,,,,,,,
157,Tripel,4,True,True,True,True,0.988,0.976,0.980,0.652,0.992
234,Blond,4,True,True,True,True,0.984,0.980,0.976,0.412,0.976
103,Stout/Porter,4,True,True,True,True,0.980,0.984,0.984,0.728,0.980
10,Low/No alcohol,4,True,True,True,True,0.976,1.000,1.000,0.396,1.000
101,Blond,3,True,False,True,True,1.000,0.988,0.988,0.348,0.988
223,Low/No alcohol,3,True,True,False,True,0.996,0.948,0.952,0.980,0.920
150,Brett/cofermented,3,True,False,True,True,0.992,0.996,0.996,0.784,0.984
192,Brut,2,False,False,True,True,0.972,0.992,0.992,0.552,0.996
168,Scotch,1,False,True,False,False,0.968,0.968,0.972,0.640,0.960


In [17]:
# 距离证据热图：仅展示至少一种方法族标记的样本
plot_cols = [f"{c}_percentile" for c in score_columns]
plot_labels = ["Shrinkage MD", "PCA-kNN", "LOF", "Correlation", "Log-ratio profile"]
heatmap_data = distance_review[plot_cols].copy()
heatmap_data.columns = plot_labels
heatmap_data.index = [f"{beer} | {distance_review.loc[beer, 'style']}" for beer in heatmap_data.index]

fig_height = max(4.5, 0.36 * len(heatmap_data))
fig, ax = plt.subplots(figsize=(10, fig_height))
sns.heatmap(heatmap_data, cmap="YlOrRd", vmin=0.80, vmax=1.0,
            annot=True, fmt=".3f", linewidths=0.4, cbar_kws={"label": "经验百分位"}, ax=ax)
ax.set_title("高维距离异常证据")
ax.set_xlabel("")
ax.set_ylabel("beer | style")
plt.tight_layout()
distance_figure = FIGURE_DIR / "02_outlier_distance_consensus_v1.png"
fig.savefig(distance_figure, dpi=180, bbox_inches="tight")
plt.show()

distance_scores.to_csv(INTERIM_DIR / "chem_outlier_distance_scores_v1.csv", encoding="utf-8-sig")
assert hashlib.sha256((INTERIM_DIR / "chem_log10_matrix_v1.csv").read_bytes()).hexdigest() == v1_manifest["sha256"]
print("距离分析完成；v1 数值矩阵未改变。")


距离分析完成；v1 数值矩阵未改变。


### 3.2 异常样本的主要驱动变量

对至少两个独立距离家族同时标记的样本，按**风格中心化、稳健缩放后的有向偏差**给变量排序：绝对值决定排名，正负方向在热图中展示。排序使用的量和热图展示的量是同一个，不叠加多种贡献分数。

结果同时回连文件1、文件8、单位、变换关系、变量类型和回填标记，便于人工核查每个驱动变量的原始来源。

In [18]:
consensus_ids = distance_scores.index[distance_scores["distance_consensus_flag"]]
consensus_ids = distance_scores.loc[consensus_ids].sort_values(
    ["evidence_families", "shrinkage_md2_percentile"], ascending=False).index

col_position = {col: j for j, col in enumerate(multivariate_cols)}
beer_position = {beer: i for i, beer in enumerate(outlier_data.index)}

# 驱动变量只用一个指标：风格中心化稳健偏差的绝对值（与热图展示的量一致）
driver_rows = []
for beer in consensus_ids:
    i = beer_position[beer]
    per_feature = pd.DataFrame({
        "beer": beer,
        "style": style_v1.loc[beer],
        "column": multivariate_cols,
        "scaled_deviation": X_scaled[i],
    })
    per_feature["rank_within_beer"] = per_feature["scaled_deviation"].abs() \
        .rank(method="first", ascending=False).astype(int)
    driver_rows.append(per_feature)

outlier_drivers = pd.concat(driver_rows, ignore_index=True)
outlier_drivers["block"] = outlier_drivers["column"].map(feature_audit["block"])
outlier_drivers["value_v1"] = [outlier_data.loc[b, c] for b, c in zip(outlier_drivers["beer"], outlier_drivers["column"])]
outlier_drivers["file1_value"] = [chem_values.loc[b, c] for b, c in zip(outlier_drivers["beer"], outlier_drivers["column"])]
outlier_drivers["file8_value"] = [
    pd.to_numeric(pd.Series([ab.loc[b, c]]), errors="coerce").iloc[0] if c in ab.columns else np.nan
    for b, c in zip(outlier_drivers["beer"], outlier_drivers["column"])
]
outlier_drivers["file8_unit"] = outlier_drivers["column"].map(unit_of)
outlier_drivers["transform_relation"] = outlier_drivers["column"].map(relation)
outlier_drivers["imputed"] = [bool(imputed_mask.loc[b, c]) for b, c in zip(outlier_drivers["beer"], outlier_drivers["column"])]
outlier_drivers["univariate_flag"] = [
    bool(((cell_outlier_flags["beer"] == b) & (cell_outlier_flags["column"] == c)).any())
    for b, c in zip(outlier_drivers["beer"], outlier_drivers["column"])
]

top_drivers = outlier_drivers[outlier_drivers["rank_within_beer"] <= 10].sort_values(
    ["beer", "rank_within_beer"])
display(top_drivers[["beer", "style", "rank_within_beer", "column", "block", "scaled_deviation",
                     "value_v1", "file1_value", "file8_value", "file8_unit",
                     "transform_relation", "imputed", "univariate_flag"]])


,beer,style,rank_within_beer,column,block,scaled_deviation,value_v1,file1_value,file8_value,file8_unit,transform_relation,imputed,univariate_flag
701,10,Low/No alcohol,1,X4VG,GC-MS 校准化合物,29.910633,-0.460560,-0.460560,0.34629,mg/L,log10,False,False
809,10,Low/No alcohol,2,beta.cyclohomocitral,GC-MS 相对丰度,-15.978034,-1.817819,-1.817819,NaN,NaN,NaN,False,False
874,10,Low/No alcohol,3,alpha.muurolol,GC-MS 相对丰度,-14.040184,-1.817819,-1.817819,NaN,A.U.,NaN,False,False
880,10,Low/No alcohol,4,tau.muurolol,GC-MS 相对丰度,-12.719827,-1.817819,-1.817819,NaN,NaN,NaN,False,False
718,10,Low/No alcohol,5,X1.propanol,GC-MS 相对丰度,-11.327213,-1.817819,-1.817819,NaN,NaN,NaN,False,False
884,10,Low/No alcohol,6,X2.3.dihydro.farnesol,GC-MS 相对丰度,-10.463468,-1.817819,-1.817819,NaN,NaN,NaN,False,False
851,10,Low/No alcohol,7,germacrene.D,GC-MS 相对丰度,-10.161527,-1.817819,-1.817819,NaN,NaN,NaN,False,False
770,10,Low/No alcohol,8,isoamyl.lactate,GC-MS 相对丰度,-10.129182,-1.817819,-1.817819,NaN,NaN,NaN,False,False
803,10,Low/No alcohol,9,hydroxy.methyl.furfural,GC-MS 相对丰度,-10.031412,-1.817819,-1.817819,NaN,NaN,NaN,False,False
714,10,Low/No alcohol,10,ethanol..v.v.,标准理化指标,-9.869769,-1.154902,-1.154902,0.07000,v/v,log10,False,False


In [19]:
# 每个样本的驱动结构摘要
driver_summary_rows = []
for beer in consensus_ids:
    top = top_drivers[top_drivers["beer"].eq(beer)]
    top5 = top.nsmallest(5, "rank_within_beer")
    driver_summary_rows.append({
        "beer": beer,
        "style": style_v1.loc[beer],
        "evidence_families": int(distance_scores.loc[beer, "evidence_families"]),
        "top_5_drivers": "; ".join(top5["column"]),
        "top10_calibrated_n": int(top["block"].isin(["GC-MS 校准化合物", "GC-FID/FPD 目标定量"]).sum()),
        "top10_relative_n": int(top["block"].eq("GC-MS 相对丰度").sum()),
        "top10_unflagged_unit_n": int(top["block"].eq("文件8无校准标记").sum()),
        "top10_imputed_n": int(top["imputed"].sum()),
        "top10_univariate_confirmed_n": int(top["univariate_flag"].sum()),
    })

driver_summary = pd.DataFrame(driver_summary_rows).set_index("beer")
display(driver_summary)

# 选每个异常样本排名前3的变量，展示其有方向的风格中心化稳健偏差
heat_columns = list(dict.fromkeys(
    top_drivers[top_drivers["rank_within_beer"] <= 3].sort_values(
        ["beer", "rank_within_beer"])["column"].tolist()))
heat_values = pd.DataFrame(index=consensus_ids, columns=heat_columns, dtype=float)
for beer in consensus_ids:
    i = beer_position[beer]
    for col in heat_columns:
        heat_values.loc[beer, col] = X_scaled[i, col_position[col]]
heat_values.index = [f"{beer} | {style_v1.loc[beer]}" for beer in heat_values.index]

fig_width = max(12, 0.55 * len(heat_columns))
fig, ax = plt.subplots(figsize=(fig_width, 6))
limit = max(4.5, float(np.nanquantile(np.abs(heat_values.to_numpy()), 0.95)))
sns.heatmap(heat_values, cmap="vlag", center=0, vmin=-limit, vmax=limit,
            linewidths=0.35, cbar_kws={"label": "风格中心化稳健偏差"}, ax=ax)
ax.set_title("共识异常样本的主要驱动变量")
ax.set_xlabel("")
ax.set_ylabel("beer | style")
plt.xticks(rotation=55, ha="right")
plt.tight_layout()
driver_figure = FIGURE_DIR / "02_outlier_driver_heatmap_v1.png"
fig.savefig(driver_figure, dpi=180, bbox_inches="tight")
plt.show()

outlier_drivers.to_csv(INTERIM_DIR / "chem_outlier_driver_details_v1.csv", index=False, encoding="utf-8-sig")
driver_summary.to_csv(INTERIM_DIR / "chem_outlier_driver_summary_v1.csv", encoding="utf-8-sig")
assert hashlib.sha256((INTERIM_DIR / "chem_log10_matrix_v1.csv").read_bytes()).hexdigest() == v1_manifest["sha256"]
print("驱动变量检查完成；v1 数值矩阵未改变。")


,style,evidence_families,top_5_drivers,top10_calibrated_n,top10_relative_n,top10_unflagged_unit_n,top10_imputed_n,top10_univariate_confirmed_n
beer,,,,,,,,
157,Tripel,4,n.decanol; X4.methyl.1.pentanol; n.dodecanol; ...,0,10,0,0,0
234,Blond,4,beta.humulene; n.decanol; X4.methyl.1.pentanol...,1,9,0,0,0
103,Stout/Porter,4,n.decanol; X4.methyl.1.pentanol; n.dodecanol; ...,0,10,0,0,0
10,Low/No alcohol,4,X4VG; beta.cyclohomocitral; alpha.muurolol; ta...,1,8,0,0,0
101,Blond,3,beta.humulene; n.decanol; n.dodecanol; n.nonan...,1,9,0,0,0
223,Low/No alcohol,3,ethyl.phenylacetate; ethyl.hexadecanoate; etha...,0,9,0,0,0
150,Brett/cofermented,3,n.decanol; n.dodecanol; n.nonanol; X3.methylth...,0,10,0,0,0
192,Brut,2,n.decanol; X4.methyl.1.pentanol; n.dodecanol; ...,0,10,0,0,0


驱动变量检查完成；v1 数值矩阵未改变。


### 3.3 检出谱型核查：区分“浓度异常”与“峰检出谱型异常”

数每个共识样本在 142 个有共同底限的变量中落在底限的数量，与同风格同伴比较：未检出峰异常偏多，说明整张谱的检出能力有问题（谱型异常），不应解读为个别浓度超标；未检出数量不极端，才更可能是常规的浓度异常。风格样本数 < 8 时百分位解释力有限，单独标注。

In [20]:
# 共同底限数量核查：区分“浓度异常”与“峰检出谱型异常”
shared_floor_cols = feature_audit.index[feature_audit["has_shared_floor"]]
shared_floor_count = pd.Series(
    np.isclose(outlier_data[shared_floor_cols].to_numpy(), -1.81781854, atol=1e-7).sum(axis=1),
    index=outlier_data.index,
)

detection_pattern_rows = []
for beer in consensus_ids:
    peer_count = shared_floor_count[style_v1.eq(style_v1.loc[beer])]
    detection_pattern_rows.append({
        "beer": beer,
        "style": style_v1.loc[beer],
        "style_n": len(peer_count),
        "shared_floor_n": int(shared_floor_count.loc[beer]),
        "detected_among_142": int(len(shared_floor_cols) - shared_floor_count.loc[beer]),
        "within_style_floor_percentile": float((peer_count <= shared_floor_count.loc[beer]).mean()),
        "style_floor_min": int(peer_count.min()),
        "style_floor_median": float(peer_count.median()),
        "style_floor_max": int(peer_count.max()),
    })

detection_pattern_summary = pd.DataFrame(detection_pattern_rows).set_index("beer")
detection_pattern_summary["pattern_interpretation"] = np.select(
    [detection_pattern_summary["within_style_floor_percentile"] >= 0.95,
     detection_pattern_summary["within_style_floor_percentile"] <= 0.05],
    ["同风格中未检出峰偏多", "同风格中检出峰偏多"],
    default="未检出数量本身不极端",
)
detection_pattern_summary["small_style_warning"] = detection_pattern_summary["style_n"] < 8

univariate_consensus = cell_outlier_flags[cell_outlier_flags["beer"].isin(consensus_ids)].copy()
univariate_consensus["overall_lower_percentile"] = [
    float((outlier_data[c] <= v).mean()) for c, v in zip(univariate_consensus["column"], univariate_consensus["value"])
]
univariate_consensus["overall_upper_tail"] = [
    float((outlier_data[c] >= v).mean()) for c, v in zip(univariate_consensus["column"], univariate_consensus["value"])
]

display(detection_pattern_summary)
display(univariate_consensus.sort_values(["beer", "robust_z"], ascending=[True, False]))

detection_pattern_summary.to_csv(INTERIM_DIR / "chem_outlier_detection_pattern_v1.csv", encoding="utf-8-sig")
univariate_consensus.to_csv(INTERIM_DIR / "chem_outlier_univariate_confirmations_v1.csv", index=False, encoding="utf-8-sig")


,style,style_n,shared_floor_n,detected_among_142,within_style_floor_percentile,style_floor_min,style_floor_median,style_floor_max,pattern_interpretation,small_style_warning
beer,,,,,,,,,,
157,Tripel,28,102,40,1.000000,9,26.5,102,同风格中未检出峰偏多,False
234,Blond,31,88,54,0.967742,9,23.0,94,同风格中未检出峰偏多,False
103,Stout/Porter,13,101,41,1.000000,15,24.0,101,同风格中未检出峰偏多,False
10,Low/No alcohol,8,49,93,1.000000,27,35.0,49,同风格中未检出峰偏多,False
101,Blond,31,94,48,1.000000,9,23.0,94,同风格中未检出峰偏多,False
223,Low/No alcohol,8,34,108,0.500000,27,35.0,49,未检出数量本身不极端,False
150,Brett/cofermented,6,86,56,1.000000,10,23.0,86,同风格中未检出峰偏多,True
192,Brut,4,96,46,1.000000,20,28.5,96,同风格中未检出峰偏多,True


,beer,style,column,block,value,reference,robust_z,global_robust_z,direction,imputed,overall_lower_percentile,overall_upper_tail
84,101,Blond,ethyl.phenylacetate,GC-MS 相对丰度,4.547854,within_style_and_global,-6.002330,-5.109807,low,False,0.012,0.992
82,101,Blond,E.ocimenone,GC-MS 相对丰度,3.310341,within_style_and_global,-6.253891,-4.933239,low,False,0.020,0.984
107,101,Blond,gamma.nonalactone,GC-MS 校准化合物,-2.261316,within_style_and_global,-7.421999,-6.177705,low,False,0.024,0.980
118,101,Blond,X.E..beta.damascenone,GC-MS 校准化合物,-1.946349,within_style_and_global,-10.025855,-6.326198,low,False,0.020,0.984
80,103,Stout/Porter,ethyl.benzoate,GC-MS 相对丰度,3.925501,within_style_and_global,-4.716057,-4.922154,low,False,0.012,0.992
71,103,Stout/Porter,n.octanol,GC-MS 相对丰度,4.772160,within_style_and_global,-5.092867,-5.739919,low,False,0.008,0.996
137,103,Stout/Porter,germacrene.D,GC-MS 相对丰度,2.383575,within_style_and_global,-5.331252,-4.533367,low,False,0.088,0.916
124,103,Stout/Porter,X.E..beta.damascenone,GC-MS 校准化合物,-2.229765,within_style_and_global,-12.954487,-7.379799,low,False,0.008,0.996
113,103,Stout/Porter,gamma.nonalactone,GC-MS 校准化合物,-3.638879,within_style_and_global,-18.178649,-10.677997,low,False,0.004,1.000
63,157,Tripel,benzyl.alcohol,GC-MS 校准化合物,-6.540608,within_style_and_global,-5.140166,-4.697401,low,False,0.020,0.984


In [21]:
# 检出谱型可视化：共识样本的未检出峰数量 vs 同风格同伴
fig, ax = plt.subplots(figsize=(10, 5.8))

y_labels = []
for y_pos, beer in enumerate(consensus_ids):
    # 同伴不含样本自身，作为对照分布
    peer_count = shared_floor_count[style_v1.eq(style_v1.loc[beer])].drop(beer, errors="ignore")
    ax.scatter(peer_count, np.full(len(peer_count), y_pos),
               s=28, alpha=0.6, color="#9AA0A6", edgecolors="none",
               label="同风格同伴" if y_pos == 0 else None)

    own_count = int(shared_floor_count.loc[beer])
    pct = detection_pattern_summary.loc[beer, "within_style_floor_percentile"]
    ax.scatter([own_count], [y_pos], s=95, color="#D62728", zorder=3,
               label="共识异常样本" if y_pos == 0 else None)
    ax.annotate(f"{own_count} (风格内 P{pct:.0%})",
                (own_count, y_pos), xytext=(9, 0), textcoords="offset points",
                va="center", fontsize=9)

    style_n = int(detection_pattern_summary.loc[beer, "style_n"])
    warn = " *" if bool(detection_pattern_summary.loc[beer, "small_style_warning"]) else ""
    y_labels.append(f"{beer} | {style_v1.loc[beer]} (n={style_n}){warn}")

ax.set_yticks(range(len(consensus_ids)))
ax.set_yticklabels(y_labels)
ax.invert_yaxis()
ax.set_xlabel(f"{len(shared_floor_cols)} 个有共同底限的变量中，落在底限的数量")
ax.set_title("共识样本的检出谱型：未检出峰数量与同风格同伴对比（* 风格样本 < 8）", pad=30)
ax.margins(x=0.12)
# 图例放在绘图区外上方，避免遮挡底部行的标注
ax.legend(frameon=False, loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2)
plt.tight_layout()
detection_figure = FIGURE_DIR / "02_outlier_detection_pattern_v1.png"
fig.savefig(detection_figure, dpi=180, bbox_inches="tight")
plt.show()

assert hashlib.sha256((INTERIM_DIR / "chem_log10_matrix_v1.csv").read_bytes()).hexdigest() == v1_manifest["sha256"]
print("检出谱型图已保存；v1 数值矩阵未改变。")

检出谱型图已保存；v1 数值矩阵未改变。


### 3.4 统一异常标记

所有异常证据收束为每个样本一个 `final_flag`，同时命中多档时取最高档：

1. **最高级**：PCA 与 Isolation Forest 同时命中，或 ≥3 个证据家族一致；
2. **风格异常**：共识样本且未检出峰在同风格里极端偏多——整谱检出能力的问题，不按浓度异常解读；
3. **一般异常**：其余多变量提示，或仅有单变量标记（后者多为变量层尺度问题，按伪影候选对待）；
4. **不认为异常**：无任何标记。

`suggested_action` 只给建议、不执行任何剔除；剔除/降权留给建模阶段的敏感性分析。

In [22]:
# ===== 统一异常标记：把所有证据收束到每个样本一档 =====
unified = pd.DataFrame({
    "beer": sample_outlier_scores.index,
    "style": sample_outlier_scores["style"].to_numpy(),
    "univariate_flag_count": sample_outlier_scores["univariate_flag_count"].to_numpy(),
    "pca_structure_flag": sample_outlier_scores["pca_structure_flag"].to_numpy(),
    "isolation_flag": sample_outlier_scores["isolation_flag"].to_numpy(),
    "evidence_families": distance_scores["evidence_families"].reindex(sample_outlier_scores.index).to_numpy(),
    "distance_consensus_flag": distance_scores["distance_consensus_flag"].reindex(sample_outlier_scores.index).to_numpy(),
}).set_index("beer")

# 共识样本才有驱动变量和谱型解读；其余填空
unified["top_5_drivers"] = driver_summary["top_5_drivers"].reindex(unified.index).fillna("")
unified["pattern_interpretation"] = detection_pattern_summary["pattern_interpretation"].reindex(unified.index).fillna("")

# 分档：注意括号——& 的优先级高于比较运算符
is_top = ((unified["pca_structure_flag"] & unified["isolation_flag"])
          | (unified["distance_consensus_flag"] & (unified["evidence_families"] >= 3)))
is_pattern = (unified["distance_consensus_flag"]
              & unified["pattern_interpretation"].eq("同风格中未检出峰偏多"))
is_any = (unified["pca_structure_flag"] | unified["isolation_flag"]
          | (unified["evidence_families"] >= 1)
          | (unified["univariate_flag_count"] > 0))

unified["final_flag"] = np.select(
    [is_top, is_pattern & ~is_top, is_any],
    ["最高级", "风格异常", "一般异常"],
    default="不认为异常",
)

action_map = {
    "最高级": "人工复核优先；建模敏感性必做",
    "风格异常": "谱型问题：敏感性对比或关注底限变量，不按浓度异常解读",
    "一般异常": "保留；单变量为主者按变量层伪影候选",
    "不认为异常": "保留",
}
unified["suggested_action"] = unified["final_flag"].map(action_map)

flag_order = ["最高级", "风格异常", "一般异常", "不认为异常"]
unified["final_flag"] = pd.Categorical(unified["final_flag"], categories=flag_order, ordered=True)

tier_summary = unified["final_flag"].value_counts().reindex(flag_order).rename("n").to_frame()
tier_summary["pct"] = (tier_summary["n"] / len(unified) * 100).round(1)
display(tier_summary)

display(unified.sort_values(
    ["final_flag", "evidence_families", "univariate_flag_count"],
    ascending=[True, False, False]).head(20))

unified.to_csv(INTERIM_DIR / "chem_outlier_unified_flags_v1.csv", encoding="utf-8-sig")
assert hashlib.sha256((INTERIM_DIR / "chem_log10_matrix_v1.csv").read_bytes()).hexdigest() == v1_manifest["sha256"]
print("统一异常标记已保存：chem_outlier_unified_flags_v1.csv；v1 数值矩阵未改变。")

,n,pct
final_flag,,
最高级,7,2.8
风格异常,1,0.4
一般异常,78,31.2
不认为异常,164,65.6


,style,univariate_flag_count,pca_structure_flag,isolation_flag,evidence_families,distance_consensus_flag,top_5_drivers,pattern_interpretation,final_flag,suggested_action
beer,,,,,,,,,,
157,Tripel,6,True,True,4,True,n.decanol; X4.methyl.1.pentanol; n.dodecanol; ...,同风格中未检出峰偏多,最高级,人工复核优先；建模敏感性必做
103,Stout/Porter,5,True,True,4,True,n.decanol; X4.methyl.1.pentanol; n.dodecanol; ...,同风格中未检出峰偏多,最高级,人工复核优先；建模敏感性必做
234,Blond,4,True,False,4,True,beta.humulene; n.decanol; X4.methyl.1.pentanol...,同风格中未检出峰偏多,最高级,人工复核优先；建模敏感性必做
10,Low/No alcohol,0,True,False,4,True,X4VG; beta.cyclohomocitral; alpha.muurolol; ta...,同风格中未检出峰偏多,最高级,人工复核优先；建模敏感性必做
101,Blond,4,False,False,3,True,beta.humulene; n.decanol; n.dodecanol; n.nonan...,同风格中未检出峰偏多,最高级,人工复核优先；建模敏感性必做
223,Low/No alcohol,0,True,False,3,True,ethyl.phenylacetate; ethyl.hexadecanoate; etha...,未检出数量本身不极端,最高级,人工复核优先；建模敏感性必做
150,Brett/cofermented,0,False,False,3,True,n.decanol; n.dodecanol; n.nonanol; X3.methylth...,同风格中未检出峰偏多,最高级,人工复核优先；建模敏感性必做
192,Brut,0,False,True,2,True,n.decanol; X4.methyl.1.pentanol; n.dodecanol; ...,同风格中未检出峰偏多,风格异常,谱型问题：敏感性对比或关注底限变量，不按浓度异常解读
162,Lambic,1,False,False,1,False,,,一般异常,保留；单变量为主者按变量层伪影候选


统一异常标记已保存：chem_outlier_unified_flags_v1.csv；v1 数值矩阵未改变。


## 4. 固定版本的观测状态
待办：进入模型后的敏感性分析

固定的 `chem_log10_matrix_v1.csv` 不再修改。另存与其同形状的观测状态矩阵：

- `0 = observed`：正常观测；
- `1 = left_censored`：共同检测底限，或文件1缺失但文件8明确为0的未检出值；
- `2 = derived_imputed`：派生变量 `aroma_hops_sum` 原始缺失并经验证模型回填。

敏感性分析所需的三种左删失矩阵和异常样本证据继续保留，但推迟到具体预测模型完成后再运行，不再回改预处理版本。


In [23]:
# 观测状态矩阵只描述来源，不改变固定数值
STATUS_OBSERVED = 0
STATUS_LEFT_CENSORED = 1
STATUS_DERIVED_IMPUTED = 2

observation_status = pd.DataFrame(
    STATUS_OBSERVED, index=chem_values.index, columns=chem_values.columns, dtype="uint8")

# 文件1中已经写成共同底限的值
for col in feature_audit.index[feature_audit["has_shared_floor"]]:
    floor_cells = np.isclose(chem_values[col], floor_value[col], atol=1e-8)
    observation_status.loc[floor_cells, col] = STATUS_LEFT_CENSORED

# 文件1为NA、文件8为0的60个单元格，包括无共同底限的A.U.列
observation_status[nd_mask] = STATUS_LEFT_CENSORED

# aroma_hops_sum 原始缺失且由模型回填
observation_status.loc[missing_idx, "aroma_hops_sum"] = STATUS_DERIVED_IMPUTED

status_dictionary = {
    "0": "observed：正常观测值",
    "1": "left_censored：低于检测限/共同底限，不是真缺失",
    "2": "derived_imputed：派生变量原始缺失，经验证模型回填",
}
status_counts = observation_status.stack().value_counts().sort_index().rename("cells").to_frame()
status_counts["meaning"] = [status_dictionary[str(i)] for i in status_counts.index]
display(status_counts)

observation_status.to_csv(INTERIM_DIR / "chem_observation_status_v1.csv")
(INTERIM_DIR / "chem_observation_status_v1_dictionary.json").write_text(
    json.dumps(status_dictionary, ensure_ascii=False, indent=2), encoding="utf-8")

sensitivity_manifest = {
    "status": "deferred_until_prediction_models_exist",
    "fixed_primary_matrix": "chem_log10_matrix_v1.csv",
    "left_censoring_scenarios": {
        "half_min": "chem_log10_matrix_v1.csv",
        "lod_sqrt2_proxy": "chem_log10_matrix_lod_sqrt2_proxy_v1.csv",
        "qrilc": "chem_log10_matrix_qrilc_v1.csv",
    },
    "sample_review_files": [
        "chem_outlier_distance_scores_v1.csv",
        "chem_outlier_unified_flags_v1.csv",
        "chem_outlier_detection_pattern_v1.csv",
    ],
    "planned_comparisons": [
        "主模型在三种左删失方案下的性能和特征稳定性",
        "全部样本与排除稀疏谱复核样本后的性能差异",
    ],
    "note": "敏感性分析不修改v1，只在模型结果存在后比较结论稳定性。",
}
(INTERIM_DIR / "chem_sensitivity_manifest_v1.json").write_text(
    json.dumps(sensitivity_manifest, ensure_ascii=False, indent=2), encoding="utf-8")

assert observation_status.shape == final_matrix.shape
assert set(np.unique(observation_status)) <= {0, 1, 2}
assert hashlib.sha256((INTERIM_DIR / "chem_log10_matrix_v1.csv").read_bytes()).hexdigest() == v1_manifest["sha256"]



,cells,meaning
0,50870,observed：正常观测值
1,6867,left_censored：低于检测限/共同底限，不是真缺失
2,13,derived_imputed：派生变量原始缺失，经验证模型回填


观测状态已分离保存；v1数值矩阵未改变；敏感性分析已登记为延期任务。